In [2]:
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import linprog
import sympy as sp
import itertools

In [121]:
def find_hyperplane_hypercube_intersections(a, b, l_bounds, u_bounds):
    # Number of dimensions
    n = len(a)
    
    # Create symbolic variables for each dimension
    x = sp.symbols(f'x:{n}')
    
    # Hyperplane equation
    hyperplane_eq = sum(a[i] * x[i] for i in range(n)) - b
    
    solutions = []
    invalid_solutions = 0
    # Iterate over each dimension 
    for i in range(n):

        # Iterate through combinations of bounds for other dimensions
        all_combination = itertools.product(*[(l_bounds[j], u_bounds[j]) if j != i else (None,) for j in range(n)])
        for prod in all_combination:

            subs_eq = hyperplane_eq
            subs_dict = {}
            
            # Substitute bounds for all dimensions except the current one
            for j in range(n):
                if j != i:
                    subs_dict[x[j]] = prod[j]
            
            # Substitute
            subs_eq = subs_eq.subs(subs_dict) 
            
            # Solve the equation for the current dimension
            solution = sp.solve(subs_eq, x[i])
            
            # Check if the solution is within the bounds and add it if it is
            for s in solution:
                if s.is_real and l_bounds[i] <= s <= u_bounds[i]: # and a @ s >= b?
                    valid_point = [prod[k] if k != i else s for k in range(n)]
                    solutions.append(tuple(valid_point))
                else:
                    invalid_solutions += 1
    
    # Remove duplicates
    solutions = list(set(solutions))
    print(f"Found {len(solutions)} valid solutions and {invalid_solutions} invalid solutions.")
    return solutions

In [122]:
def linear_subproblem(c, a, b, l, u):
    n = len(c)
    x_star = []

    for i in range(n):
        if c[i] >= 0:
            x_star.append(l[i])
        else:
            x_star.append(u[i])
    
    if (a @ x_star) >= b:
        return x_star
    
    print("Not in the corner")
    intercections = find_hyperplane_hypercube_intersections(a, b, l, u)

    # Calculate the objective function value for each intersection point
    fun_values = [c @ np.array(inter) for inter in intercections]

    # Find the intersection point that minimizes the objective function
    min_index = np.argmin(fun_values)
    return intercections[min_index]

In [124]:
# Coefficients of the objective function
c = np.array([333, 11.2, 300, 100, 234, 212, 400, 177, 300, 120])

# Coefficients matrix for inequality constraints (ax >= b)
a = np.array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
A = -np.array([a])

b = 9000
b_ = -np.array(b)

l = np.array([15, 9.8, 13, 32, 12, 3, 2, 10, 12, 10])
u = np.array([856, 1000, 1000, 3890, 1500, 1002, 3990, 1290, 2000, 1433])
# Bounds for the variables
bounds = [] 
for i in range(len(l)):
    bounds.append((l[i], u[i]))

In [125]:
# 2n^(n-1)
solution = linear_subproblem(c, a, b, l, u)
solution

Not in the corner
Found 1114 valid solutions and 4006 invalid solutions.


(15.0, 1000, 13.0, 3890, 343.000000000000, 1002, 2.0, 1290, 12.0, 1433)

In [126]:
# Solving the linear problem
res = linprog(c, A_ub=A, b_ub=b_, bounds=bounds, method='highs')

res.x, res.success, res.message

(array([1.500e+01, 1.000e+03, 1.300e+01, 3.890e+03, 3.430e+02, 1.002e+03,
        2.000e+00, 1.290e+03, 1.200e+01, 1.433e+03]),
 True,
 'Optimization terminated successfully. (HiGHS Status 7: Optimal)')